# 🔍 Notebook 1: Agentic RAG con Gemini + LangChain

## ¿Qué es Agentic RAG?

En el **RAG tradicional (2-step)**, la recuperación de documentos ocurre **siempre** antes de la generación. Es predecible pero rígido.

En **Agentic RAG**, un agente (impulsado por un LLM) **decide por sí mismo** cuándo y cómo recuperar información. Puede:
- Consultar múltiples fuentes de forma selectiva
- Combinar herramientas de búsqueda
- Razonar paso a paso antes de buscar

```
Usuario pregunta
     ↓
  Agente razona: ¿necesito buscar?
     ↓              ↓
   Sí → usa tool    No → responde directo
     ↓
  Resultado → Agente genera respuesta
```

**Referencia:** https://docs.langchain.com/oss/python/langchain/retrieval#agentic-rag

## 📦 Instalación de dependencias

In [12]:
# !pip install -q langchain langchain-google-genai langgraph langchain-community chromadb

## 🔑 Configuración de API Key

In [13]:
from dotenv import load_dotenv
import os

load_dotenv()
API_KEY = os.getenv("GEMINI_API_KEY")

# API key de Gemini
# API_KEY = userdata.get('GEMINI_API_KEY')

In [14]:
# import os
# from google.colab import userdata  # Si usas Colab

# # En Colab: guarda tu clave en Secrets con el nombre GOOGLE_API_KEY
# # os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

# # O directamente (no recomendado en producción):
# os.environ["GOOGLE_API_KEY"] = "TU_GOOGLE_API_KEY_AQUÍ"

## 📚 Paso 1: Crear una base de conocimiento simple

In [15]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_classic.schema import Document

from langchain_huggingface.embeddings import HuggingFaceEmbeddings

In [16]:
# Documentos de ejemplo sobre el máster
documentos = [
    Document(
        page_content="LangChain es un framework para construir aplicaciones con LLMs. "
                     "Permite encadenar componentes como modelos, herramientas y memoria.",
        metadata={"source": "langchain_intro", "tema": "frameworks"}
    ),
    Document(
        page_content="RAG (Retrieval-Augmented Generation) combina búsqueda de documentos "
                     "con generación de texto. Permite al LLM acceder a información externa.",
        metadata={"source": "rag_intro", "tema": "rag"}
    ),
    Document(
        page_content="LangGraph permite crear agentes con estado como grafos dirigidos. "
                     "Es ideal para flujos de trabajo complejos con múltiples pasos.",
        metadata={"source": "langgraph_intro", "tema": "agentes"}
    ),
    Document(
        page_content="Los embeddings son representaciones vectoriales de texto. "
                     "Permiten comparar similitud semántica entre documentos y consultas.",
        metadata={"source": "embeddings_intro", "tema": "embeddings"}
    ),
    Document(
        page_content="AWS Bedrock es un servicio de Amazon para acceder a modelos fundacionales "
                     "de IA generativa a través de una API unificada y segura.",
        metadata={"source": "aws_bedrock", "tema": "cloud"}
    ),
]

# Crear embeddings con Gemini
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-2")

# embeddings = HuggingFaceEmbeddings(
#     model_name="sentence-transformers/all-MiniLM-L6-v2"
# )

# Crear vector store en memoria
vectorstore = Chroma.from_documents(
    documents=documentos,
    embedding=embeddings,
    collection_name="master_ia_docs"
)

print(f"✅ Base de conocimiento creada con {len(documentos)} documentos")

✅ Base de conocimiento creada con 5 documentos


## 🛠️ Paso 2: Crear la herramienta de búsqueda (RAG Tool)

In [17]:
from langchain.tools import tool

retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

@tool
def buscar_documentacion(query: str) -> str:
    """Busca información relevante en la documentación del máster de IA.
    Úsala cuando necesites información sobre LangChain, RAG, LangGraph, 
    embeddings, o AWS Bedrock."""
    docs = retriever.invoke(query)
    if not docs:
        return "No encontré información relevante sobre ese tema."
    
    resultados = []
    for i, doc in enumerate(docs, 1):
        fuente = doc.metadata.get('source', 'desconocida')
        resultados.append(f"[Fuente: {fuente}]\n{doc.page_content}")
    
    return "\n\n".join(resultados)

# Probar la herramienta directamente
resultado = buscar_documentacion.invoke({"query": "¿Qué es RAG?"})
print("📄 Resultado de búsqueda directa:")
print(resultado)

📄 Resultado de búsqueda directa:
[Fuente: rag_intro]
RAG (Retrieval-Augmented Generation) combina búsqueda de documentos con generación de texto. Permite al LLM acceder a información externa.

[Fuente: embeddings_intro]
Los embeddings son representaciones vectoriales de texto. Permiten comparar similitud semántica entre documentos y consultas.


## 🤖 Paso 3: Crear el Agente RAG con Gemini

In [20]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent

# Inicializar el modelo Gemini
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    temperature=0
)

# Prompt del sistema
system_prompt = """Eres un asistente experto en IA Generativa para un máster universitario.
Tienes acceso a documentación del máster. Cuando el usuario pregunta algo técnico, 
SIEMPRE usa la herramienta `buscar_documentacion` para fundamentar tu respuesta.
Si la pregunta es de conversación general, puedes responder directamente.
Responde siempre en español."""

# Crear el agente RAG (ReAct = Reasoning + Acting)
agente_rag = create_agent(
    model=llm,
    tools=[buscar_documentacion],
    system_prompt=system_prompt
)

print("✅ Agente RAG creado correctamente")

✅ Agente RAG creado correctamente


## 💬 Paso 4: Probar el Agente RAG

Observa cómo el agente **decide** si necesita buscar o no.

In [21]:
from langchain_core.messages import HumanMessage

def preguntar_al_agente(pregunta: str):
    """Función helper para hacer preguntas al agente y ver el proceso."""
    print(f"\n{'='*60}")
    print(f"👤 Pregunta: {pregunta}")
    print('='*60)
    
    respuesta = agente_rag.invoke({
        "messages": [HumanMessage(content=pregunta)]
    })
    
    # Mostrar el proceso paso a paso
    print("\n🔄 Proceso del agente:")
    for msg in respuesta["messages"]:
        tipo = msg.__class__.__name__
        if tipo == "HumanMessage":
            print(f"  👤 Usuario: {msg.content[:100]}")
        elif tipo == "AIMessage":
            if msg.tool_calls:
                for tc in msg.tool_calls:
                    print(f"  🔧 Agente decide usar herramienta: {tc['name']}")
                    print(f"     Query: {tc['args'].get('query', '')}")
            else:
                print(f"\n  🤖 Respuesta final:\n  {msg.content}")
        elif tipo == "ToolMessage":
            print(f"  📚 Resultado recuperado: {msg.content[:150]}...")

# Pregunta 1: Requiere búsqueda
preguntar_al_agente("¿Qué es LangGraph y para qué sirve?")


👤 Pregunta: ¿Qué es LangGraph y para qué sirve?

🔄 Proceso del agente:
  👤 Usuario: ¿Qué es LangGraph y para qué sirve?
  🔧 Agente decide usar herramienta: buscar_documentacion
     Query: LangGraph
  📚 Resultado recuperado: [Fuente: langgraph_intro]
LangGraph permite crear agentes con estado como grafos dirigidos. Es ideal para flujos de trabajo complejos con múltiples pa...

  🤖 Respuesta final:
  


In [22]:
# Pregunta 2: También requiere búsqueda
preguntar_al_agente("Explícame la diferencia entre RAG y embeddings")


👤 Pregunta: Explícame la diferencia entre RAG y embeddings

🔄 Proceso del agente:
  👤 Usuario: Explícame la diferencia entre RAG y embeddings
  🔧 Agente decide usar herramienta: buscar_documentacion
     Query: diferencia entre RAG y embeddings
  📚 Resultado recuperado: [Fuente: embeddings_intro]
Los embeddings son representaciones vectoriales de texto. Permiten comparar similitud semántica entre documentos y consulta...

  🤖 Respuesta final:
  


In [23]:
# Pregunta 3: No requiere búsqueda (conversación general)
preguntar_al_agente("¿Cuántos días tiene una semana?")


👤 Pregunta: ¿Cuántos días tiene una semana?

🔄 Proceso del agente:
  👤 Usuario: ¿Cuántos días tiene una semana?

  🤖 Respuesta final:
  Una semana tiene 7 días.


## 🧪 Ejercicio propuesto

1. **Añade más documentos** al vectorstore (por ejemplo, sobre Prompt Engineering o Pipelines ML)
2. **Crea una segunda herramienta** que busque en una fuente diferente (puede simular una búsqueda web)
3. **Observa cómo el agente decide** qué herramienta usar según la pregunta

```python
# Pista para añadir documentos:
nuevo_doc = Document(
    page_content="El Prompt Engineering es el arte de diseñar instrucciones...",
    metadata={"source": "prompt_engineering", "tema": "prompts"}
)
vectorstore.add_documents([nuevo_doc])
```

## 📝 Puntos clave para recordar

| Concepto | Descripción |
|----------|-------------|
| **Agentic RAG** | El agente decide *cuándo* y *cómo* recuperar información |
| **Tool** | Función que el agente puede llamar para obtener información externa |
| **ReAct** | Patrón de razonamiento: Reason → Act → Observe → Repeat |
| **Retriever** | Componente que busca documentos relevantes en el vectorstore |
| **Embeddings** | Representaciones vectoriales usadas para buscar por similitud semántica |